# Module 11 - 실습 2: Self-Reflection 루프

## 학습 목표
- LangGraph로 Self-Reflection 루프를 구현할 수 있다
- generate → reflect → 조건부 재생성 패턴을 이해한다
- 최대 반복 횟수로 무한 루프를 방지할 수 있다

## 참조 자료
- 학습 문서: `docs/part5-advanced/11-quality-assurance.md` (섹션 2.3)

## 개념 요약

```
Self-Reflection 루프:

[generate] → [reflect] → (approved?) → [finalize]
    ↑              │ (not approved & count < max)
    └──────────────┘

* 최대 반복 횟수를 반드시 제한! (무한 루프 방지)
```

FakeLLM으로 실제 LLM 없이 테스트합니다.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import re
from typing import TypedDict
from langgraph.graph import StateGraph, END
from common.fake_llm import FakeLLM

print("Self-Reflection 실습 준비 완료")

MAX_REFLECTION_ROUNDS = 2

## 실습 1: ReflectionState 정의

Self-Reflection 에이전트의 상태를 정의하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: topic: str - 작성 주제
# 힌트 2: draft: str - 현재 초안
# 힌트 3: reflection: dict | None - 검토 결과 {"approved": bool, "score": int, "feedback": str}
# 힌트 4: reflection_count: int - 검토 횟수
# 힌트 5: final_output: str | None - 최종 결과

class ReflectionState(TypedDict):
    pass  # TODO: 5개 필드 정의

## 실습 2: 노드 구현

generate, reflect, finalize 노드를 FakeLLM을 사용하여 구현하세요.

In [ ]:
# FakeLLM 설정
# generate 노드용 - 초안 생성
generate_llm = FakeLLM(responses={
    "피드백": "피드백을 반영하여 수정된 내용입니다. asyncio는 비동기 프로그래밍의 핵심입니다.",
    "default": "asyncio는 Python의 비동기 프로그래밍 라이브러리입니다. async/await 키워드를 사용합니다.",
})

# reflect 노드용 - 초안 검토 (처음에는 미승인, 두 번째에는 승인)
reflect_responses = [
    '{"approved": false, "score": 6, "feedback": "더 구체적인 예시가 필요합니다."}',
    '{"approved": true, "score": 9, "feedback": ""}',
]
reflect_call_count = [0]  # 호출 횟수 추적

def get_reflect_response():
    idx = min(reflect_call_count[0], len(reflect_responses) - 1)
    reflect_call_count[0] += 1
    return reflect_responses[idx]

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: generate_node는 reflection이 있으면 피드백을 참조하여 수정, 없으면 새로 생성
# 힌트 2: reflect_node는 초안을 검토하여 {"approved": bool, "score": int, "feedback": str} 반환
# 힌트 3: reflection_count를 1씩 증가
# 힌트 4: finalize_node는 최종 결과를 포맷하여 반환

def generate_node(state: ReflectionState) -> dict:
    """초안을 생성하거나 피드백을 반영하여 수정합니다."""
    # TODO: 구현
    pass

def reflect_node(state: ReflectionState) -> dict:
    """초안을 검토합니다."""
    # TODO: get_reflect_response()로 검토 결과 얻기
    # TODO: json.loads()로 파싱
    # TODO: reflection_count 증가
    pass

def decide_after_reflection(state: ReflectionState) -> str:
    """검토 결과에 따라 다음 경로를 결정합니다."""
    # TODO: approved 또는 count >= MAX → "finalize"
    # TODO: 미승인 → "generate"
    pass

def finalize_node(state: ReflectionState) -> dict:
    """최종 결과를 확정합니다."""
    # TODO: f"[검토 {count}회 완료, 점수: {score}]\n\n{draft}" 형태로 반환
    pass

## 실습 3: Self-Reflection 그래프 구성

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: StateGraph(ReflectionState)로 그래프 생성
# 힌트 2: generate → reflect → (decide_after_reflection) → generate 또는 finalize
# 힌트 3: add_conditional_edges로 순환 엣지 구현

def build_reflection_graph():
    """Self-Reflection 루프 그래프를 구성합니다."""
    graph = StateGraph(ReflectionState)
    
    # TODO: 노드 추가
    # TODO: 엣지 연결
    # TODO: 조건부 엣지 추가 (순환!)
    
    return graph.compile()

app = build_reflection_graph()

In [ ]:
# 리셋 후 실행
reflect_call_count[0] = 0

result = app.invoke({
    "topic": "Python asyncio",
    "draft": "",
    "reflection": None,
    "reflection_count": 0,
    "final_output": None,
})

print(f"검토 횟수: {result['reflection_count']}")
print(f"최종 결과:\n{result['final_output']}")

In [ ]:
# 검증 셀
assert result["final_output"] is not None, "final_output이 None입니다"
assert result["reflection_count"] > 0, "최소 1회 검토가 필요합니다"
assert result["reflection_count"] <= MAX_REFLECTION_ROUNDS + 1, \
    f"최대 반복 횟수를 초과했습니다. 현재: {result['reflection_count']}"
assert "검토" in result["final_output"], "최종 결과에 검토 횟수가 포함되어야 합니다"
print(f"✅ 실습 3 완료! Self-Reflection 루프가 {result['reflection_count']}회 실행되었습니다.")

## 정리

이번 실습에서 배운 내용:
1. `add_conditional_edges`로 순환 엣지를 구현할 수 있다
2. `reflection_count`로 최대 반복 횟수를 제한하여 무한 루프를 방지한다
3. 피드백이 있으면 반영하고, 승인되거나 한도 초과 시 종료한다

## 다음 모듈
- **실습 3**: `03_golden_set.ipynb` - 골든셋 평가